# Notebook 49: Connectors

This notebook demonstrates external-system connector patterns using `multigen.connectors`.
All examples use mock implementations — no real API keys are required.

Topics covered:
- `WebhookRouter` — glob-pattern event routing
- `LoggingConnector` + `ConnectorRegistry` — register, send, broadcast
- `EventDrivenBranchManager` — conditional branching on event context
- `DocumentIngester` — chunk text for RAG pipelines
- `IngestionPipeline` — ingest + top_context with keyword ranking

In [ ]:
import sys
sys.path.insert(0, '../sdk')


## Mock Implementation of `multigen.connectors`

In [ ]:
import fnmatch
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Tuple

print('Imports ready.')


## Section 1 — WebhookRouter

In [ ]:
@dataclass
class DispatchResult:
    event_type: str
    matched_handlers: List[str]
    payloads: List[Any]


class WebhookRouter:
    def __init__(self):
        self._routes: List[Tuple[str, str, Callable]] = []  # (pattern, name, handler)

    def register(self, pattern: str, name: str, handler: Callable):
        self._routes.append((pattern, name, handler))
        print(f'  Registered handler {name!r} for pattern {pattern!r}')

    def dispatch(self, event_type: str, payload: Any) -> DispatchResult:
        matched = []
        results = []
        for pattern, name, handler in self._routes:
            if fnmatch.fnmatch(event_type, pattern):
                matched.append(name)
                results.append(handler(payload))
        return DispatchResult(event_type=event_type, matched_handlers=matched, payloads=results)


router = WebhookRouter()

print('Registering handlers:')
router.register('crm.*',          'crm_logger',   lambda p: f'CRM log: {p["type"]}')
router.register('order.shipped',  'notify_user',  lambda p: f'Notify user {p.get("user_id")}')
router.register('order.*',        'order_audit',  lambda p: f'Audit order event: {p["type"]}')
router.register('*',              'global_trace', lambda p: f'Trace: {p["type"]}')

events = [
    ('crm.lead_created',  {'type': 'crm.lead_created',  'lead_id': 99}),
    ('order.shipped',     {'type': 'order.shipped',     'user_id': 42, 'order_id': 7}),
    ('order.cancelled',   {'type': 'order.cancelled',   'order_id': 8}),
    ('payment.received',  {'type': 'payment.received',  'amount': 99.99}),
]

print()
for event_type, payload in events:
    result = router.dispatch(event_type, payload)
    print(f'Event "{event_type}":')
    print(f'  matched_handlers : {result.matched_handlers}')
    for p in result.payloads:
        print(f'    -> {p}')


## Section 2 — LoggingConnector + ConnectorRegistry

In [ ]:
class LoggingConnector:
    """Simple connector that captures all payloads sent to it."""

    def __init__(self, name: str):
        self.name = name
        self.captured: List[Any] = []

    def send(self, payload: Any):
        self.captured.append(payload)
        print(f'  [{self.name}] received: {payload}')


class ConnectorRegistry:
    def __init__(self):
        self._connectors: Dict[str, LoggingConnector] = {}

    def register(self, connector: LoggingConnector):
        self._connectors[connector.name] = connector

    def send(self, name: str, payload: Any):
        if name not in self._connectors:
            raise KeyError(f'Unknown connector: {name}')
        self._connectors[name].send(payload)

    def broadcast(self, payload: Any):
        for c in self._connectors.values():
            c.send(payload)


slack_conn  = LoggingConnector('slack')
pager_conn  = LoggingConnector('pagerduty')
audit_conn  = LoggingConnector('audit_log')

registry = ConnectorRegistry()
registry.register(slack_conn)
registry.register(pager_conn)
registry.register(audit_conn)

print('Sending to slack only:')
registry.send('slack', {'msg': 'Deploy completed', 'env': 'prod'})

print('\nBroadcasting to all connectors:')
registry.broadcast({'alert': 'High error rate detected', 'value': 0.12})

print(f'\nCapture counts: slack={len(slack_conn.captured)}  '
      f'pagerduty={len(pager_conn.captured)}  audit_log={len(audit_conn.captured)}')


## Section 3 — EventDrivenBranchManager

In [ ]:
@dataclass
class BranchResult:
    triggered: List[str]
    not_triggered: List[str]


class EventDrivenBranchManager:
    """Register named conditions; evaluate a context dict against all conditions."""

    def __init__(self):
        self._conditions: List[Tuple[str, Callable[[Dict], bool]]] = []

    def register(self, name: str, condition: Callable[[Dict], bool]):
        self._conditions.append((name, condition))
        print(f'  Registered branch condition: {name!r}')

    def evaluate(self, ctx: Dict) -> BranchResult:
        triggered = []
        not_triggered = []
        for name, cond in self._conditions:
            if cond(ctx):
                triggered.append(name)
            else:
                not_triggered.append(name)
        return BranchResult(triggered=triggered, not_triggered=not_triggered)


print('Registering branch conditions:')
branch_mgr = EventDrivenBranchManager()
branch_mgr.register('high_risk',   lambda ctx: ctx.get('risk_score', 0) > 0.7)
branch_mgr.register('low_quality', lambda ctx: ctx.get('quality_score', 1) < 0.5)
branch_mgr.register('vip_user',    lambda ctx: ctx.get('user_tier') == 'vip')

contexts = [
    {'risk_score': 0.85, 'quality_score': 0.40, 'user_tier': 'standard'},
    {'risk_score': 0.30, 'quality_score': 0.80, 'user_tier': 'vip'},
    {'risk_score': 0.20, 'quality_score': 0.90, 'user_tier': 'standard'},
]

print()
for i, ctx in enumerate(contexts, 1):
    result = branch_mgr.evaluate(ctx)
    print(f'Context {i}: {ctx}')
    print(f'  triggered     : {result.triggered}')
    print(f'  not_triggered : {result.not_triggered}')
    print()


## Section 4 — DocumentIngester

In [ ]:
@dataclass
class TextChunk:
    index: int
    text: str
    start: int
    end: int


class DocumentIngester:
    def __init__(self, chunk_size: int = 200, overlap: int = 50):
        self.chunk_size = chunk_size
        self.overlap = overlap
        self._chunks: List[TextChunk] = []

    def ingest(self, text: str):
        self._chunks = []
        step = self.chunk_size - self.overlap
        idx = 0
        chunk_num = 0
        while idx < len(text):
            end = min(idx + self.chunk_size, len(text))
            self._chunks.append(TextChunk(
                index=chunk_num, text=text[idx:end], start=idx, end=end
            ))
            idx += step
            chunk_num += 1
            if end == len(text):
                break

    @property
    def chunk_count(self) -> int:
        return len(self._chunks)

    def to_context(self, separator: str = '\n---\n') -> str:
        return separator.join(c.text for c in self._chunks)


# Generate a 1000-character document
doc_text = ('The quick brown fox jumps over the lazy dog. ' * 24)[:1000]
print(f'Document length: {len(doc_text)} chars')

ingester = DocumentIngester(chunk_size=200, overlap=50)
ingester.ingest(doc_text)

print(f'chunk_size : {ingester.chunk_size}')
print(f'overlap    : {ingester.overlap}')
print(f'chunk_count: {ingester.chunk_count}')

print('\nChunk overview:')
for c in ingester._chunks:
    print(f'  chunk {c.index}: chars [{c.start}:{c.end}]  len={len(c.text)}')

print('\nto_context() (first 200 chars):')
print(ingester.to_context()[:200])


## Section 5 — IngestionPipeline

In [ ]:
class IngestionPipeline:
    """Ingests a document and retrieves top-k chunks by keyword relevance."""

    def __init__(self, chunk_size: int = 200, overlap: int = 50):
        self._ingester = DocumentIngester(chunk_size=chunk_size, overlap=overlap)

    def ingest(self, text: str):
        self._ingester.ingest(text)
        print(f'Ingested {len(text)} chars -> {self._ingester.chunk_count} chunks')

    def _score(self, chunk: TextChunk, query: str) -> float:
        """Simple keyword overlap score."""
        query_words = set(query.lower().split())
        chunk_words = set(chunk.text.lower().split())
        overlap = len(query_words & chunk_words)
        return round(overlap / max(len(query_words), 1), 4)

    def top_context(self, query: str, top_k: int = 3) -> List[Dict]:
        ranked = sorted(
            self._ingester._chunks,
            key=lambda c: self._score(c, query),
            reverse=True,
        )
        return [
            {'chunk_index': c.index, 'score': self._score(c, query),
             'text_preview': c.text[:60]}
            for c in ranked[:top_k]
        ]


# Use a more varied document for better demo
varied_doc = (
    'Artificial intelligence is transforming every industry. '
    'Machine learning models learn from large datasets. '
    'Natural language processing enables computers to understand text. '
    'Deep learning uses neural networks with many layers. '
    'Reinforcement learning trains agents through reward signals. '
    'Computer vision allows machines to interpret images. '
    'Generative models can create new content from training data. '
    'Large language models are trained on vast amounts of text. '
    'Transformers are the dominant architecture for NLP tasks. '
    'Fine-tuning adapts pretrained models to specific domains. '
    'Retrieval augmented generation combines search and generation. '
    'Agents can use tools to interact with external systems. '
    'Multi-agent systems coordinate multiple AI agents. '
    'Memory systems help agents recall previous interactions. '
) * 2

pipeline = IngestionPipeline(chunk_size=150, overlap=30)
pipeline.ingest(varied_doc)

query = 'neural networks deep learning layers'
print(f'\nQuery: "{query}"')
print('Top 3 chunks by relevance:')
for item in pipeline.top_context(query, top_k=3):
    print(f'  chunk {item["chunk_index"]:02d}: score={item["score"]}  preview={item["text_preview"]!r}')
